In [2]:
import nltk
import string # importa il modulo string, che contiene una collezione di stringhe come string.punctuation per la gestione della punteggiatura

from nltk import punkt # importa il modulo punkt da NLTK, che è utilizzato per la tokenizzazione delle frasi
from nltk.stem import WordNetLemmatizer # importa la classe WordNetLemmatizer da NLTK, che è utilizzata per la lemmatizzazione delle parole
from nltk.corpus import wordnet # importa il corpus wordnet da NLTK, che è una grande base di dati lessicale di inglese
from collections import Counter # da una libreria python si importa la classe Counter per contare le frequenze
from transformers import pipeline # importa la funzione pipeline dalla libreria transformers, che è utilizzata per creare pipeline di elaborazione del linguaggio naturale
from transformers import AutoTokenizer, AutoModelForSequenceClassification # importa le classi AutoTokenizer e AutoModelForSequenceClassification dalla libreria transformers, che sono utilizzate per caricare modelli pre-addestrati e tokenizzatori

nltk.download('wordnet')
nltk.download('punkt_tab') # scarica un modello di tokenizzazione più efficace
nltk.download('averaged_perceptron_tagger_eng') # scarica il tagger POS per l'inglese



[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [3]:
from google.colab import files
uploaded = files.upload()

Saving corpora_1.txt to corpora_1.txt
Saving corpora_2.txt to corpora_2.txt


In [4]:
c1 = 'corpora_1.txt' # assegnamento alle variabili c1 e c2 dei file da analizzare
c2 = 'corpora_2.txt'
with open (c1,"r") as infile: # apertura dei file in modalità lettura
  contenuti_c1 = infile.read() # il contenuto dei file viene memorizzato nelle variabili
with open (c2,"r") as infile:
  contenuti_c2 = infile.read()

# sentence splitting
from nltk.tokenize import sent_tokenize # dalla libreria NLTK si è presa la funzione sent_tokenize
sentences_c1 = sent_tokenize(contenuti_c1) # la funzione prende in input i file e li suddivide in frasi
sentences_c2 = sent_tokenize(contenuti_c2) # le frasi ottenute vengono poi salvate nelle variabili sentence_c1 e sentence_c2
print("sentences_corpus_1: ", sentences_c1)
print("sentences_corpus_2: ", sentences_c2)

# tokenizzazione
from nltk.tokenize import word_tokenize # dalla libreria NLTK si prende la funzione word_tokenize
tokens_c1 = word_tokenize(contenuti_c1) # la funzione prende in input i file e li suddivide in token
tokens_c2 = word_tokenize(contenuti_c2) # i risultati vengono salvati nelle variabili tokens_c1 e tokens_c2
print("tokens_corpus_1: ", tokens_c1)
print("tokens_corpus_2: ", tokens_c2)

# pos tagging
pos_c1 = nltk.tag.pos_tag(tokens_c1) # la funzione prende in input la lista di token di ogni file e restituisce una lista di tuple (word, POS tag) salavete nelle variabili pos_c1 e pos_c2
pos_c2 = nltk.tag.pos_tag(tokens_c2) # questa funzione assegna dei tag ad ogni token in base alla loro categoria grammaticale
print("pos_corpus_1: ", pos_c1)
print("pos_corpus_2: ", pos_c2)

# lemmatizzazione con pos
def get_wordnet_pos(treebank_tag): # definizione della funzione get_wordnet_pos per convertire i POS tags in un formato che WordNetLemmatizer può capire
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

# creazione di un oggetto lemmatizer per convertire ogni parola nel suo corrispondente lemma
lemmatizer = WordNetLemmatizer()
# creazione di una list comprehension che itera su ogni elemento in pos_c1 (lista di tuple) e per ogni tupla (word, pos)
# la word viene convertita nel suo lemma corrispondente richiamando al metodo lemmatize dell'oggetto lemmatizer
# mentre la funzione get_wordnet_pos serve a convertire il tag del pos in un formato compatibile con il WordNet.
lemmatized_c1 = [lemmatizer.lemmatize(word, get_wordnet_pos(pos)) for word, pos in pos_c1]
lemmatized_c2 = [lemmatizer.lemmatize(word, get_wordnet_pos(pos)) for word, pos in pos_c2]
print("lemmatized_corpus_1: ", lemmatized_c1)
print("lemmatized_corpus_2: ", lemmatized_c2)

# confronto corpora per numero di frasi
if len(sentences_c1) > len(sentences_c2):
  print(f"Il primo corpus ha {len(sentences_c1) - len(sentences_c2)} frasi in più del secondo corpus.")
elif len(sentences_c1) < len(sentences_c2):
  print(f"Il secondo corpus ha {len(sentences_c2) - len(sentences_c1)} frasi in più del primo corpus.")

# confronto corpora per numero di token
if len(tokens_c1) > len(tokens_c2):
  print(f"Il primo corpus ha {len(tokens_c1) - len(tokens_c2)} token in più del secondo corpus.")
elif len(tokens_c1) < len(tokens_c2):
  print(f"Il secondo corpus ha {len(tokens_c2) - len(tokens_c1)} token in più del primo corpus.")

# confrnto corpora per lunghezza media delle frasi (senza contare la punteggiatura)

# creazione di una comprensione di lista annidata che tokenizza ogni frase in
# sentences_c1 e sentences_c2 e salvando in lista solo i token che non sono
# punteggiatura
frasi_senza_punteggiatura_c1 = [[word for word in word_tokenize(sentence) if word not in string.punctuation] for sentence in sentences_c1]
frasi_senza_punteggiatura_c2 = [[word for word in word_tokenize(sentence) if word not in string.punctuation] for sentence in sentences_c2]
# calcolo della lunghezza media delle frasi sommando la lunghezza di ogni frase
# in frasi_senza_punteggiatura e poi dividendo per la lunghezza totale
len_media_frasi_c1 = (sum(len(frasi) for frasi in frasi_senza_punteggiatura_c1)) / len(frasi_senza_punteggiatura_c1)
len_media_frasi_c2 = (sum(len(frasi) for frasi in frasi_senza_punteggiatura_c2)) / len(frasi_senza_punteggiatura_c2)
if len_media_frasi_c1 > len_media_frasi_c2:
  print(f"Il primo corpus ha frasi più corte di quelle del secondo, con una lunghezza media di {len_media_frasi_c1} token.")
elif len_media_frasi_c1 < len_media_frasi_c2:
  print(f"Il secondo corpus ha frasi più corte di quelle del primo, con una lunghezza media di {len_media_frasi_c2} token.")
else:
  print(f"I due corpus hanno frasi di lunghezza media uguali a {len_media_frasi_c1} token.")

# confronto corpora per lunghezza media dei token (senza contare la punteggiatura)
token_senza_punteggiatura_c1 = [word for word in tokens_c1 if word not in string.punctuation]
token_senza_punteggiatura_c2 = [word for word in tokens_c2 if word not in string.punctuation]
len_media_token_c1 = (sum(len(token) for token in token_senza_punteggiatura_c1)) / len(token_senza_punteggiatura_c1)
len_media_token_c2 = (sum(len(token) for token in token_senza_punteggiatura_c2)) / len(token_senza_punteggiatura_c2)
if len_media_token_c1 > len_media_token_c2:
  print(f"Il primo corpus ha token più lunghi di quelli del secondo, con una lunghezza media di {len_media_token_c1} token.")
elif len_media_token_c1 < len_media_token_c2:
  print(f"Il secondo corpus ha token più lunghi di quelli del primo, con una lunghezza media di {len_media_token_c2} token.")
else:
    print(f"I due corpus hanno token di lunghezza media uguali a {len_media_token_c1} token.")

# confronto corpora rispetto alla distribuzione delle pos nei primi 1000 token
pos_c1_1000 = nltk.tag.pos_tag(token_senza_punteggiatura_c1[:1000]) # selezione delle pos per i primi 1000 token
                                                                    # i risultati sono poi salvati nelle variabili pos_c1_1000 e pos_c2_1000
pos_c2_1000 = nltk.tag.pos_tag(token_senza_punteggiatura_c2[:1000])
tags_only_c1 = [pos for word, pos in pos_c1_1000] # a partire dalla lista di tuple (word, pos) si estraggono solo i pos
tags_only_c2 = [pos for word, pos in pos_c2_1000] # i risultati sono poi salvati nelle variabili tags_only_c1 e tags_only_c2
frequenze_pos_c1 = Counter(tags_only_c1) # si conta il numero di volte che ogni pos appare nelle liste tags_only_c1 e tags_only_c2
frequenze_pos_c2 = Counter(tags_only_c2) # Counter è una classe della libreria collections di Python che conta le frequenze di ciascun
                                         # elemento in una iterabile (=lista)
print("frequenze_pos_corpus_1: ", frequenze_pos_c1)
print("frequenze_pos_corpus_2: ", frequenze_pos_c2)

# confronto corpora per dimensione del vocabolario
vocabolario_c1 = set(token_senza_punteggiatura_c1) # a partire dai token di ciascun corpus, costruisco i vocabolari
                                # corrispondenti selezionado solo le parole uniche senza duplicati
vocabolario_c2 = set(token_senza_punteggiatura_c2)
lunghezza_vocabolario_c1 = len(vocabolario_c1) # con len si contano tutte le parole del vocabolario e poi si
                                               # fanno i vari confronti
lunghezza_vocabolario_c2 = len(vocabolario_c2)
if lunghezza_vocabolario_c1 > lunghezza_vocabolario_c2:
  print(f"Il primo corpus ha un vocabolario più grande, con una lunghezza di vocabolario uguale a {lunghezza_vocabolario_c1}.")
elif lunghezza_vocabolario_c1 < lunghezza_vocabolario_c2:
  print(f"Il secondo corpus ha un vocabolario più grande, con una lunghezza di vocabolario uguale a {lunghezza_vocabolario_c2}.")
else:
  print(f"I due corpus hanno lo stesso vocabolario, con una lunghezza di vocabolario uguale a {lunghezza_vocabolario_c1}.")

# confronto corpora per ttr su porzioni incrementali di 200 token
def ttr(tokens, porzione = 200): # definizione della funzione ttr che ha come parametri i tokens su cui calcolare la ttr e la porzione
                                 # incrementale su cui applicare la ttr che rimane 200 di default
  valori_ttr = []                # creazione di una lista vuota su cui si aggiungono i risultati delle ttr per ogni porzione di 200 tokens
  for i in range(0, len(tokens), porzione): # iterazione a partire dal primo token che ha indice 0 fino a raggiungere la lunghezza totale dei tokens
                                            # porzione definisce la dimensione della finestra, cioè 200 tokens per finestra
    porzione_tokens = tokens[i:i+porzione]  # ad ogni iterazione si seleziona una finestra di tokens che vanno dal token in indice i
                                            # fino a i+porzione escluso (per quello i+)
    ttr_result = len(set(porzione_tokens)) / len(porzione_tokens)
    valori_ttr.append(ttr_result)
  return valori_ttr

ttr_c1 = ttr(token_senza_punteggiatura_c1, porzione = 200)
ttr_c2 = ttr(token_senza_punteggiatura_c2, porzione = 200)
print("ttr_corpus_1: ", ttr_c1)
print("ttr_corpus_2: ", ttr_c2)

# confronto corpora per numero di lemmi distinti
vocabolario_lemmi_c1 = len(set(lemmatized_c1)) # a partire dai lemma estratti precedentemente,
                                               # con set elimino i duplicati e conto solo le occorrenze uniche di lemma
vocabolario_lemmi_c2 = len(set(lemmatized_c2))
if vocabolario_lemmi_c1 > vocabolario_lemmi_c2:
  print(f"Il primo corpus ha un vocabolario di lemmi distinti più grande, con una lunghezza di vocabolario di lemmi distinti uguale a {vocabolario_lemmi_c1}.")
elif vocabolario_lemmi_c1 < vocabolario_lemmi_c2:
  print(f"Il secondo corpus ha un vocabolario di lemmi distinti più grande, con una lunghezza di vocabolario di lemmi distinti uguale a {vocabolario_lemmi_c2}.")
else:
  print(f"I due corpus hanno lo stesso vocabolario di lemmi distinti, con una lunghezza di vocabolario di lemmi distinti uguale a {vocabolario_lemmi_c1}.")

# confronto corpora per numero medio di lemmi per frase

# la seguente funzione itera su ogni frase in sentences: per ogni frase si esegue tokenizzazione, pos tagging e lemmatizzazione
# successivamente calcola la lunghezza media dei lemmi per frase, sommando ogni lemma di ciascuna frase e rapportando la somma al numero totale di frasi nei corpus
def calcola_numero_medio_lemmi_per_frase(sentences):
    return sum(len([lemmatizer.lemmatize(word, get_wordnet_pos(pos)) for word, pos in nltk.tag.pos_tag(word_tokenize(sentence))]) for sentence in sentences) / len(sentences)
numero_medio_lemmi_per_frase_c1 = calcola_numero_medio_lemmi_per_frase(sentences_c1)
numero_medio_lemmi_per_frase_c2 = calcola_numero_medio_lemmi_per_frase(sentences_c2)
if numero_medio_lemmi_per_frase_c1 > numero_medio_lemmi_per_frase_c2:
  print(f"Il primo corpus ha un numero medio di lemmi per frase più alto, con un numero medio di lemmi per frase uguale a {numero_medio_lemmi_per_frase_c1}.")
elif numero_medio_lemmi_per_frase_c1 < numero_medio_lemmi_per_frase_c2:
  print(f"Il secondo corpus ha un numero medio di lemmi per frase più alto, con un numero medio di lemmi per frase uguale a {numero_medio_lemmi_per_frase_c2}.")
else:
    print(f"I due corpus hanno lo stesso numero medio di lemmi per frase, con un numero medio di lemmi per frase uguale a {numero_medio_lemmi_per_frase_c1}.")

# confronto corpora per distribuzione di frasi positive e negative
# si è utilizzato il seguente modello pre-addestrato: https://huggingface.co/mdhugol/indonesia-bert-sentiment-classification
pretrained= "mdhugol/indonesia-bert-sentiment-classification"

# il modello e il tokenizer vengono caricati utilizzando la funzione from_pretrained
model = AutoModelForSequenceClassification.from_pretrained(pretrained)
tokenizer = AutoTokenizer.from_pretrained(pretrained)

# viene creata una pipeline di sentiment analysis utilizzando il modello e il tokenizer caricati
sentiment_analysis = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

# il modello ha creato anche un dizionario che che mappa le etichette numeriche del modello alle etichette testuali
label_index = {'LABEL_0': 'positive', 'LABEL_1': 'neutral', 'LABEL_2': 'negative'}

# a partire dal codice fornito su hugging face, ho creato la seguente funzione che prende in input le frasi
# e per ogni frase viene eseguita l'analisi di sentiment tramite pipeline; al risultato si associa poi una etichetta
# del dizionario label_index e infine i vari risultati vengono salvati in una lista di tuple contenenti la frase, l'etichetta e il punteggio
def analyze_sentiment_for_sentences(sentences):
    results = []
    for sentence in sentences:
        polarity = sentiment_analysis(sentence)
        label = label_index[polarity[0]['label']]
        score = polarity[0]['score']
        results.append((sentence, label, score))
    return results

# ho realizzato anche la seguente funzione che prende in input i risultati dell'analisi di sentiment
# e somma il numero di frasi con la stessa polarità
def calculate_distribution(polarity_results):
    positive = sum(1 for _, label, _ in polarity_results if label == 'positive')
    negative = sum(1 for _, label, _ in polarity_results if label == 'negative')
    neutral = sum(1 for _, label, _ in polarity_results if label == 'neutral')
    return positive, negative, neutral

# si procede all'analisi di sentiment delle frasi dei due corpus
polarity_sentences_c1 = analyze_sentiment_for_sentences(sentences_c1)
polarity_sentences_c2 = analyze_sentiment_for_sentences(sentences_c2)

# si calcola poi la somma di tutte le frasi con polarità uguale
positive_c1, negative_c1, neutral_c1 = calculate_distribution(polarity_sentences_c1)
positive_c2, negative_c2, neutral_c2 = calculate_distribution(polarity_sentences_c2)

# stampa dei risultati
print("Distribuzione delle frasi nel primo corpus: ")
print(f"Positive: {positive_c1}")
print(f"Negative: {negative_c1}")
print(f"Neutral: {neutral_c1}")

print("Distribuzione delle frasi nel secondo corpus: ")
print(f"Positive: {positive_c2}")
print(f"Negative: {negative_c2}")
print(f"Neutral: {neutral_c2}")

# confronto corpora per polarità complessiva utilizzando lo stesso modello pre-trained utilizzato in precedenza
pretrained= "mdhugol/indonesia-bert-sentiment-classification"

model = AutoModelForSequenceClassification.from_pretrained(pretrained)
tokenizer = AutoTokenizer.from_pretrained(pretrained)

sentiment_analysis = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

label_index = {'LABEL_0': 'positive', 'LABEL_1': 'neutral', 'LABEL_2': 'negative'}

polarity_c1 = sentiment_analysis(c1)
status = label_index[polarity_c1[0]['label']]
score = polarity_c1[0]['score']
print(f'Polarity of the first corpus --> | Label : {status} ({score * 100:.3f}%)')

polarity_c2 = sentiment_analysis(c2)
status = label_index[polarity_c2[0]['label']]
score = polarity_c2[0]['score']
print(f'Polarity of the second corpus --> | Label : {status} ({score * 100:.3f}%)')


sentences_corpus_1:  ['Neurolinguistics must be computational\nAbstract: We provide a threefold taxonomy of models in neurolinguistics: faculty models, which embrace the work of classical connectionists\nand holists and of such modern workers as Geschwind; process models, which are exemplified by the work of Luria, and which fractionate\npsycholinguistic tasks and ascribe the components to particular brain regions; and representational models, which use specific linguistic\nrepresentations to build a psycholinguistic analysis of aphasic performance.', 'We argue that further progress requires that neurolinguistics\nbecome more computational, using techniques from Artificial Intelligence to model the cooperative computation underlying language\nprocessing at a level of detail consonant with linguistic representations.', 'Finally, we note that current neurolinguistics makes virtually no contact\nwith the synapse-cell-circuit level of analysis characteristic of twentieth-century neuroscien

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Distribuzione delle frasi nel primo corpus: 
Positive: 1
Negative: 5
Neutral: 239
Distribuzione delle frasi nel secondo corpus: 
Positive: 246
Negative: 241
Neutral: 1060


Device set to use cuda:0


Polarity of the first corpus --> | Label : neutral (98.604%)
Polarity of the second corpus --> | Label : neutral (98.688%)
